## Exercise 1

**Dataset Used:** Fashion MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 11: Autoencoder
Niveau: Experten
Aufgabe 11.1: Convolutional VAE mit Bildgenerierung

Lernziel: Conv-VAE implementieren und latenten Raum erkunden
Datensatz: MNIST / Fashion MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

LATENT_DIM = 8
klassen = ['T-Shirt','Hose','Pullover','Kleid','Mantel',
           'Sandale','Hemd','Sneaker','Tasche','Stiefel']

class ConvVAE(keras.Model):
    def __init__(self, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder_net = keras.Sequential([
            keras.layers.Conv2D(32, 3, activation='relu', strides=2, padding='same'),
            keras.layers.Conv2D(64, 3, activation='relu', strides=2, padding='same'),
            keras.layers.Flatten(),
            keras.layers.Dense(256, activation='relu'),
        ])
        self.mu_layer      = keras.layers.Dense(latent_dim)
        self.logvar_layer  = keras.layers.Dense(latent_dim)
        
        # Decoder
        self.decoder_net = keras.Sequential([
            keras.layers.Dense(7*7*64, activation='relu'),
            keras.layers.Reshape((7, 7, 64)),
            keras.layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same'),
            keras.layers.Conv2DTranspose(32, 3, activation='relu', strides=2, padding='same'),
            keras.layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same'),
        ])
    
    def encode(self, x):
        h = self.encoder_net(x)
        return self.mu_layer(h), self.logvar_layer(h)
    
    def reparametrize(self, mu, logvar):
        eps = tf.random.normal(tf.shape(mu))
        return mu + tf.exp(0.5 * logvar) * eps
    
    def decode(self, z):
        return self.decoder_net(z)
    
    def call(self, x):
        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        return self.decode(z), mu, logvar
    
    def train_step(self, x):
        with tf.GradientTape() as tape:
            x_recon, mu, logvar = self(x, training=True)
            recon_loss = tf.reduce_mean(
                keras.losses.binary_crossentropy(
                    tf.reshape(x, [-1, 784]),
                    tf.reshape(x_recon, [-1, 784])
                )
            ) * 784
            kl_loss = -0.5 * tf.reduce_mean(1 + logvar - tf.square(mu) - tf.exp(logvar))
            loss = recon_loss + kl_loss
        
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {'loss': loss, 'recon': recon_loss, 'kl': kl_loss}

vae = ConvVAE(LATENT_DIM)
vae.compile(optimizer=keras.optimizers.Adam(1e-3))
vae.fit(X_train, epochs=15, batch_size=256, verbose=1)

# Interpolation zwischen zwei Klassen im Latent Space
n_interp = 10
def interpoliere(z1, z2, n):
    alphas = np.linspace(0, 1, n)
    return np.array([alpha*z2 + (1-alpha)*z1 for alpha in alphas])

# Zwei Beispiele auswählen
z1, _ = vae.encode(X_test[0:1])
z2, _ = vae.encode(X_test[100:101])

z_interp = interpoliere(z1.numpy()[0], z2.numpy()[0], n_interp)
x_interp = vae.decode(z_interp.astype('float32')).numpy()

fig, axes = plt.subplots(1, n_interp, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_interp[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle(f'Latent Space Interpolation: {klassen[y_test[0]]} → {klassen[y_test[100]]}')
plt.tight_layout()
plt.savefig('conv_vae_interpolation.png', dpi=150)
plt.show()

# Neue Bilder generieren
z_random = np.random.randn(16, LATENT_DIM).astype('float32')
generiert = vae.decode(z_random).numpy()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(generiert[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Conv-VAE: Generierte Fashion MNIST Bilder')
plt.tight_layout()
plt.savefig('conv_vae_generiert.png', dpi=150)
plt.show()


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 11: Autoencoder
Niveau: Experten
Aufgabe 11.2: Masked Autoencoder (MAE) Konzept

Lernziel: Selbstüberwachtes Lernen durch Maskierung
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

MASKEN_RATE = 0.75

def erstelle_masken(X, rate):
    """Erstellt zufällige Masken für Bilder."""
    n, h, w, c = X.shape
    X_maskiert = X.copy()
    masken = np.zeros((n, h, w, c), dtype='float32')
    
    for i in range(n):
        n_patches = h * w
        n_verdeckt = int(n_patches * rate)
        patch_idx = np.random.permutation(n_patches)[:n_verdeckt]
        rows = patch_idx // w
        cols = patch_idx % w
        X_maskiert[i, rows, cols, :] = 0.0
        masken[i, rows, cols, :] = 1.0
    
    return X_maskiert.astype('float32'), masken.astype('float32')

X_train_mask, train_masken = erstelle_masken(X_train[:10000], MASKEN_RATE)
X_test_mask,  test_masken  = erstelle_masken(X_test[:1000], MASKEN_RATE)

# MAE Modell
eingabe = keras.Input(shape=(28, 28, 1))
# Encoder
x = keras.layers.Conv2D(64, 3, activation='relu', padding='same')(eingabe)
x = keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x = keras.layers.Conv2D(32, 3, activation='relu', padding='same', strides=2)(x)
# Decoder
x = keras.layers.Conv2DTranspose(64, 3, activation='relu', padding='same', strides=2)(x)
x = keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
ausgabe = keras.layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)

mae_model = keras.Model(eingabe, ausgabe, name='MAE')
mae_model.compile(optimizer='adam', loss='mse')

# Training: maskierte Eingabe → vollständige Rekonstruktion
mae_model.fit(X_train_mask, X_train[:10000],
               epochs=15, batch_size=256, verbose=1)

# Visualisierung
X_rekonstruiert = mae_model.predict(X_test_mask[:8], verbose=0)

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for i in range(8):
    axes[0, i].imshow(X_test[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', rotation=0, labelpad=40, fontsize=8)
    
    axes[1, i].imshow(X_test_mask[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel(f'Maskiert\n({MASKEN_RATE*100:.0f}%)', rotation=0,
                               labelpad=40, fontsize=8)
    
    axes[2, i].imshow(X_rekonstruiert[i].squeeze(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('Rekonstruiert', rotation=0, labelpad=40, fontsize=8)

plt.suptitle(f'Masked Autoencoder (MAE) – {MASKEN_RATE*100:.0f}% maskiert')
plt.tight_layout()
plt.savefig('masked_autoencoder.png', dpi=150)
plt.show()

mse = np.mean((X_test[:1000] - mae_model.predict(X_test_mask, verbose=0))**2)
print(f"Rekonstruktions-MSE: {mse:.5f}")
print(f"\nMAE-Prinzip (wie in ViT-MAE von Facebook):")
print(f"- {MASKEN_RATE*100:.0f}% der Patches werden verdeckt")
print(f"- Modell lernt globale Bildstruktur zu verstehen")
print(f"- Sehr effektives selbstüberwachtes Pre-Training")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 11: Autoencoder
Niveau: Experten
Aufgabe 11.3: Beta-VAE – Disentangled Representations

Lernziel: β-VAE für entkoppelte latente Darstellungen
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

LATENT_DIM = 10

def erstelle_beta_vae(beta=1.0, name='VAE'):
    """Erstellt β-VAE mit angegebenem Beta-Wert."""
    
    # Encoder
    enc_input = keras.Input(shape=(28, 28, 1))
    x = keras.layers.Conv2D(32, 3, activation='relu', strides=2, padding='same')(enc_input)
    x = keras.layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(256, activation='relu')(x)
    mu_out      = keras.layers.Dense(LATENT_DIM)(x)
    logvar_out  = keras.layers.Dense(LATENT_DIM)(x)
    encoder = keras.Model(enc_input, [mu_out, logvar_out])
    
    # Decoder
    dec_input = keras.Input(shape=(LATENT_DIM,))
    x = keras.layers.Dense(7*7*64, activation='relu')(dec_input)
    x = keras.layers.Reshape((7, 7, 64))(x)
    x = keras.layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)
    x = keras.layers.Conv2DTranspose(32, 3, activation='relu', strides=2, padding='same')(x)
    x = keras.layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')(x)
    decoder = keras.Model(dec_input, x)
    
    class BetaVAE(keras.Model):
        def __init__(self):
            super().__init__(name=name)
            self.enc = encoder
            self.dec = decoder
            self.beta = beta
        
        def call(self, x):
            mu, logvar = self.enc(x)
            eps = tf.random.normal(tf.shape(mu))
            z = mu + tf.exp(0.5 * logvar) * eps
            return self.dec(z), mu, logvar
        
        def train_step(self, x):
            with tf.GradientTape() as tape:
                x_recon, mu, logvar = self(x, training=True)
                flat_x = tf.reshape(x, [-1, 784])
                flat_r = tf.reshape(x_recon, [-1, 784])
                recon = tf.reduce_mean(
                    keras.losses.binary_crossentropy(flat_x, flat_r)) * 784
                kl = -0.5 * tf.reduce_mean(1 + logvar - tf.square(mu) - tf.exp(logvar))
                loss = recon + self.beta * kl
            grads = tape.gradient(loss, self.trainable_variables)
            self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
            return {'loss': loss, 'recon': recon, 'kl': kl}
    
    return BetaVAE(), encoder, decoder

# Beta = 1 (Standard VAE) und Beta = 4 vergleichen
ergebnisse = {}
for beta_wert in [1.0, 4.0]:
    print(f"\nTraining β-VAE mit β={beta_wert}")
    bvae, enc, dec = erstelle_beta_vae(beta_wert, f'BetaVAE_b{int(beta_wert)}')
    bvae.compile(optimizer=keras.optimizers.Adam(1e-3))
    bvae.fit(X_train[:10000], epochs=15, batch_size=256, verbose=0)
    ergebnisse[beta_wert] = (bvae, enc, dec)

# Latente Traversal – eine Dimension variieren, andere fixiert
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
for bi, (beta_wert, (bvae, enc, dec)) in enumerate(ergebnisse.items()):
    beispiel = X_test[0:1]
    mu_val, _ = enc(beispiel)
    z_basis = mu_val.numpy()[0]
    
    n_steps = 10
    werte = np.linspace(-3, 3, n_steps)
    
    for j, v in enumerate(werte):
        z_mod = z_basis.copy()
        z_mod[0] = v  # Erste latente Dimension traversieren
        x_gen = dec(z_mod.reshape(1, -1)).numpy()[0]
        axes[bi, j].imshow(x_gen.squeeze(), cmap='gray')
        axes[bi, j].axis('off')
        if j == 0:
            axes[bi, j].set_ylabel(f'β={beta_wert}', rotation=0,
                                    labelpad=35, fontsize=10)

plt.suptitle('β-VAE Latent Traversal (Dim 0 von -3 bis +3)')
plt.tight_layout()
plt.savefig('beta_vae_traversal.png', dpi=150)
plt.show()

print("\nBei höherem β: stärkere Disentanglement")
print("Nachteil: Rekonstruktionsqualität sinkt leicht")
